# Real-Time Driving Perception Inference — Runner Notebook

Run this notebook on **Google Colab with a GPU runtime** (Runtime → Change runtime type → T4 GPU).

Each phase has its own section. Run sections top-to-bottom in a single session.

## 0. Configuration

Edit these variables before running anything else.

In [ ]:
# ── Fill in your values ──────────────────────────────────────────────────────
REPO_URL         = 'https://github.com/YOUR_USERNAME/YOUR_REPO_NAME.git'
REPO_DIR         = 'real-time-driving-inference-optimizer'   # folder cloned into /content
DRIVE_VIDEO_PATH = '/content/drive/MyDrive/clip.mp4'         # path to video on your Drive
# ─────────────────────────────────────────────────────────────────────────────

import os
WORKSPACE = f'/content/{REPO_DIR}'
print(f'Workspace : {WORKSPACE}')
print(f'Repo URL  : {REPO_URL}')

## 1. Setup — Mount Drive, Clone Repo, Install Dependencies

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

if os.path.exists(WORKSPACE):
    print('Repo already cloned — pulling latest changes...')
    !git -C {WORKSPACE} pull
else:
    !git clone {REPO_URL} {WORKSPACE}

%cd {WORKSPACE}
!pwd

In [ ]:
!pip install -q -r requirements.txt

## 2. Environment Check

In [ ]:
import torch

print(f'PyTorch version : {torch.__version__}')
print(f'CUDA available  : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU             : {torch.cuda.get_device_name(0)}')
    print(f'CUDA version    : {torch.version.cuda}')
    print(f'VRAM            : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## 3. Copy Video from Drive

In [ ]:
import shutil, os

dest = 'data/clip.mp4'
os.makedirs('data', exist_ok=True)

if not os.path.exists(dest):
    print(f'Copying {DRIVE_VIDEO_PATH} → {dest} ...')
    shutil.copy(DRIVE_VIDEO_PATH, dest)
    print('Done.')
else:
    print(f'{dest} already present.')

size_mb = os.path.getsize(dest) / 1e6
print(f'File size: {size_mb:.1f} MB')

## Phase 1 — PyTorch Baseline Inference

Runs YOLO11n on every frame. Records per-frame latency for each pipeline stage:
read → preprocess → inference → postprocess → draw+write.

Outputs:
- `results/pytorch_output.mp4` — annotated video
- `results/pytorch_raw_timings.csv` — per-frame timings

In [ ]:
!python3 scripts/run_pytorch_video.py \
    --video data/clip.mp4 \
    --output-video results/pytorch_output.mp4 \
    --results results/pytorch_raw_timings.csv \
    --model yolo11n.pt \
    --input-size 640 \
    --conf 0.25 \
    --iou 0.45 \
    --warmup 10 \
    --device cuda

### Phase 1 Results Preview

In [ ]:
import pandas as pd
import sys
sys.path.insert(0, '.')
from src.metrics import compute_stats, fps_from_mean_ms

df = pd.read_csv('results/pytorch_raw_timings.csv')
print(f'Frames recorded: {len(df)}')
print()

stages = ['read_ms', 'preprocess_ms', 'inference_ms', 'postprocess_ms', 'draw_write_ms', 'total_ms']
summary = []
for col in stages:
    s = compute_stats(df[col].tolist())
    summary.append({
        'stage':   col.replace('_ms', ''),
        'mean_ms': round(s['mean_ms'], 2),
        'p50_ms':  round(s['p50_ms'],  2),
        'p90_ms':  round(s['p90_ms'],  2),
        'p99_ms':  round(s['p99_ms'],  2),
    })

summary_df = pd.DataFrame(summary)
print(summary_df.to_string(index=False))
print()
total_mean = df['total_ms'].mean()
print(f'End-to-end FPS: {fps_from_mean_ms(total_mean):.1f}')

### Download Results

Download the CSV and annotated video before ending the session.

In [ ]:
from google.colab import files

files.download('results/pytorch_raw_timings.csv')
files.download('results/pytorch_output.mp4')

---
## Phase 2 — Benchmarking
> Coming in Phase 2 development.

## Phase 3 — ONNX Export and Validation
> Coming in Phase 3 development.

## Phase 4 — TensorRT Optimization
> Coming in Phase 4 development.

## Phase 5 — Bottleneck Optimization
> Coming in Phase 5 development.